# MLIR 编译器主线 · 第 5/8 课：Dialect Conversion 与类型转换

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：配置 legal/illegal op、conversion target 与 type converter，理解 partial/full conversion。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：编译原理基础、C++ 阅读能力
- 本课在路线中的作用：Dialect Conversion 把合法性定义与 rewrite patterns 组合，用于系统性 lower 一个 dialect；它不仅是字符串替换。

## 核心心智模型

### 1. 它是什么，解决什么问题

Dialect Conversion 把合法性定义与 rewrite patterns 组合，用于系统性 lower 一个 dialect；它不仅是字符串替换。

### 2. 它如何工作

ConversionTarget 声明最终允许的 ops；TypeConverter 改签名和 SSA 类型，materialization 连接新旧类型；conversion patterns 完成操作替换。

### 3. 正确性条件与常见误区

full conversion 结束后所有 illegal op 必须消失；函数签名、block arguments 与 call sites 必须一致转换。

### 4. 性能与工程取舍

partial conversion 便于渐进迁移但可能留下混合 IR；full conversion 边界清晰却要求覆盖所有路径。

## 具体演示

把 toy.add lower 到 arith.addf 时，若 tensor type 也变为 memref，函数参数和使用者都需同步适配。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐目标合法性：arith/func 合法，toy dialect 非法。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
ConversionTarget target(ctx);
target.addLegalDialect<______, ______>();
target.addIllegalDialect<______>();

if (failed(applyFullConversion(module, target, std::move(patterns))))
  return signalPassFailure();


### 检查方法

构造仍含一个 toy op 的输入，确认 full conversion 失败而不是静默保留。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“Dialect Conversion 与类型转换”的工作机制。

**你的答案：**


### Q2

只注册 rewrite pattern、不把源 dialect 标 illegal，会发生什么？

**你的答案：**


### Q3

何时选择 partial conversion 而不是 full conversion？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
ConversionTarget target(ctx);
target.addLegalDialect<arith::ArithDialect, func::FuncDialect>();
target.addIllegalDialect<toy::ToyDialect>();

if (failed(applyFullConversion(module, target, std::move(patterns))))
  return signalPassFailure();


### Q1 参考答案

ConversionTarget 声明最终允许的 ops；TypeConverter 改签名和 SSA 类型，materialization 连接新旧类型；conversion patterns 完成操作替换。

### Q2 参考答案

判断时先检查本课不变量：full conversion 结束后所有 illegal op 必须消失；函数签名、block arguments 与 call sites 必须一致转换。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：partial conversion 便于渐进迁移但可能留下混合 IR；full conversion 边界清晰却要求覆盖所有路径。

## 参考资料

- [Dialect Conversion](https://mlir.llvm.org/docs/DialectConversion/)
- [MLIR Toy Tutorial](https://mlir.llvm.org/docs/Tutorials/Toy/)

资料用于建立事实基线；面试回答仍需用自己的语言组织。